In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt


In [ ]:
# 1. Read CSV
df = pd.read_csv('merged_final.csv')

# 2. Basic cleaning / transformations
df['acq_date'] = pd.to_datetime(df['acq_date'], format='%Y-%m-%d', errors='coerce')
df['day_of_year'] = df['acq_date'].dt.dayofyear
df['month'] = df['acq_date'].dt.month
df['year'] = df['acq_date'].dt.year

# Drop rows with missing values if needed
df.dropna(subset=['brightness', 'NDVI'], inplace=True)

In [ ]:
# 3. Define features (X) and target (y)
#    Example: Predict 'brightness' using NDVI, lat, lon, day_of_year, etc.
features = ['latitude', 'longitude', 'NDVI', 'day_of_year', 'month', 'year']
X = df[features]
y = df['brightness']

In [ ]:
# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 5. Model Training
model = RandomForestRegressor(
    n_estimators=100, 
    max_depth=10, 
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
# 6. Predictions and Evaluation
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R^2 Score:", r2)


In [ ]:
# 7. (Optional) Feature Importance
importances = model.feature_importances_
for feat_name, imp in zip(features, importances):
    print(f"{feat_name}: {imp:.3f}")

In [ ]:
# 8. (Optional) Visualization
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('Actual Brightness')
plt.ylabel('Predicted Brightness')
plt.title('Random Forest Predictions vs Actual')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("merged_final_rescaled.csv")

# (Optional) Clamp NDVI:
df['NDVI'] = df['NDVI'].clip(-1, 1)  # or (0, 1) if your data never goes negative

# Basic feature engineering
df['acq_date'] = pd.to_datetime(df['acq_date'], errors='coerce')
df['day_of_year'] = df['acq_date'].dt.dayofyear
df['month'] = df['acq_date'].dt.month
df['year'] = df['acq_date'].dt.year

df['version'] = df['version'].astype('category')
df['version_cat'] = df['version'].cat.codes  # numeric codes

features = ['latitude', 'longitude', 'NDVI', 'day_of_year', 'month', 'year', 'version_cat']
X = df[features]
y = df['brightness']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Example hyperparameter grid for RandomizedSearchCV
param_dist = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=5,  # # of parameter settings to sample
    scoring='neg_mean_squared_error',
    cv=3,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_
print("Best Params:", random_search.best_params_)

# Evaluate
y_pred = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R^2:", r2)

# Feature importances
for feat, imp in zip(features, best_model.feature_importances_):
    print(f"{feat}: {imp:.3f}")


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_final = pd.read_csv("final.csv")

In [ ]:
# Now you can safely include 'day_of_year','month','year' in your features
features = [
    'latitude', 'longitude', 'NDVI',
    'day_of_year','month','year',
    'temp_c','dewpoint','solar_radiation','precip_mm'
]
X = df_final[features]
y = df_final['brightness']


In [ ]:
features = [
    'latitude', 'longitude', 'NDVI',
    'temp_c','dewpoint','solar_radiation','precip_mm'
]
X = df_final[features]
y = df_final['brightness']


In [ ]:
df_final['log_frp'] = np.log1p(df_final['frp'])  # log(1 + FRP)

# Define features & target
features = [
    'latitude', 'longitude', 'NDVI', 'day_of_year', 'month', 'year',
    'temp_c', 'dewpoint', 'solar_radiation', 'precip_mm'
]
X = df_final[features]
y = df_final['log_frp']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

# 1. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
rf_base = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=5,
    scoring='neg_mean_squared_error',
    cv=3,
    random_state=42
)
random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print("Best Params:", random_search.best_params_)
print("RMSE:", rmse, "R^2:", r2)

In [ ]:
print(df_final.shape)


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_final = pd.read_csv("merged_with_wind.csv")

In [ ]:
# (Re)create time-based columns if not already present
df_final['acq_date'] = pd.to_datetime(df_final['acq_date'], errors='coerce')
df_final['day_of_year'] = df_final['acq_date'].dt.dayofyear
df_final['month'] = df_final['acq_date'].dt.month
df_final['year'] = df_final['acq_date'].dt.year


In [ ]:
# Define the feature set including wind_speed
features = [
    'latitude', 'longitude', 'NDVI',
    'day_of_year', 'month', 'year',
    'temp_c', 'dewpoint', 'solar_radiation', 'precip_mm',
    'wind_speed'
]

# Target: Using log-transformed FRP (if that’s what you’re predicting)
import numpy as np
df_final['log_frp'] = np.log1p(df_final['frp'])

X = df_final[features]
y = df_final['log_frp']

In [ ]:
# Train/test split and model training
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Use the hyperparameters you found earlier
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=1,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("After adding wind speed:")
print("RMSE:", rmse)
print("R^2:", r2)

# Print feature importances to see if wind_speed is influential
for feat, imp in zip(features, rf.feature_importances_):
    print(f"{feat}: {imp:.3f}")

In [ ]:
import matplotlib.pyplot as plt

print("Wind speed descriptive statistics:")
print(df_final["wind_speed"].describe())

plt.hist(df_final["wind_speed"].dropna(), bins=50, edgecolor="black")
plt.xlabel("Wind Speed (m/s)")
plt.ylabel("Frequency")
plt.title("Distribution of Wind Speed")
plt.show()


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, precision_score, f1_score
)
import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------------
df = pd.read_csv("merged_with_wind_and_direction.csv")  # Adjust path if needed

# ---------------------------------------------------------------------------
# 2. DATA PREPARATION
# ---------------------------------------------------------------------------
# Convert acq_date to datetime and create time-based features
df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")
df["day_of_year"] = df["acq_date"].dt.dayofyear
df["month"] = df["acq_date"].dt.month
df["year"] = df["acq_date"].dt.year

# Convert temperature from Kelvin to Celsius if needed
if "temp_c" not in df.columns and "temperature" in df.columns:
    df["temp_c"] = df["temperature"] - 273.15

# Convert dewpoint from Kelvin to Celsius if average > 100
if df["dewpoint"].mean() > 100:
    df["dewpoint_c"] = df["dewpoint"] - 273.15
else:
    df["dewpoint_c"] = df["dewpoint"]

# Compute cyclic wind direction features if not already present
if "wind_dir_sin" not in df.columns or "wind_dir_cos" not in df.columns:
    df["wind_dir_sin"] = np.sin(np.deg2rad(df["wind_direction"]))
    df["wind_dir_cos"] = np.cos(np.deg2rad(df["wind_direction"]))

# Compute Relative Humidity (RH) using the Magnus formula
df["RH"] = 100 * (
    np.exp((17.625 * df["dewpoint_c"]) / (243.04 + df["dewpoint_c"])) /
    np.exp((17.625 * df["temp_c"]) / (243.04 + df["temp_c"]))
)
print("Updated Relative Humidity stats:")
print(df["RH"].describe())

# ---------------------------------------------------------------------------
# 2A. DEFINE BINARY CLASS LABEL (FRP > 30)
# ---------------------------------------------------------------------------
FRP_THRESHOLD = 30.0
df["fire_class"] = (df["frp"] > FRP_THRESHOLD).astype(int)
print(f"\nClass distribution (threshold={FRP_THRESHOLD}):")
print(df["fire_class"].value_counts())

# ---------------------------------------------------------------------------
# 3. DEFINE FEATURES & TARGET
# ---------------------------------------------------------------------------
features = [
    "latitude", "longitude", "NDVI",
    "day_of_year", "month", "year",
    "temp_c", "dewpoint_c", "solar_radiation", "precip_mm",
    "wind_speed", "wind_dir_sin", "wind_dir_cos", "RH"
]
X = df[features]
y = df["fire_class"]

# ---------------------------------------------------------------------------
# 4. TRAIN/TEST SPLIT
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_neg = sum(y_train == 0)
num_pos = sum(y_train == 1)
scale_pos_weight_value = num_neg / num_pos if num_pos > 0 else 1
print(f"\nComputed scale_pos_weight = {scale_pos_weight_value:.2f} (neg:pos)")

# ---------------------------------------------------------------------------
# 5. XGBOOST CLASSIFIER WITH GRID SEARCH (optimizing F1)
# ---------------------------------------------------------------------------
xgb_clf = xgb.XGBClassifier(random_state=42, tree_method="hist")
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [6, 10],
    "learning_rate": [0.1, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "scale_pos_weight": [
        scale_pos_weight_value * 0.5,
        scale_pos_weight_value,
        scale_pos_weight_value * 1.5
    ]
}
grid = GridSearchCV(
    xgb_clf,
    param_grid=param_grid,
    scoring="f1",  # Optimize for F1 score
    cv=3,
    verbose=1
)
print("\nStarting XGBoost grid search (optimizing F1, with scale_pos_weight)...")
grid.fit(X_train, y_train)
best_clf = grid.best_estimator_
print("Best Params:", grid.best_params_)

# ---------------------------------------------------------------------------
# 6. EVALUATE AT DEFAULT THRESHOLD (0.5)
# ---------------------------------------------------------------------------
print("\n--- RESULTS AT DEFAULT THRESHOLD (0.5) ---")
y_pred_default = best_clf.predict(X_test)
acc_default = accuracy_score(y_test, y_pred_default)
cm_default = confusion_matrix(y_test, y_pred_default)
report_default = classification_report(y_test, y_pred_default)
print(f"Accuracy: {acc_default:.4f}")
print("Confusion Matrix:")
print(cm_default)
print("\nClassification Report:")
print(report_default)

# ---------------------------------------------------------------------------
# 7. THRESHOLD TUNING (0.5 to 0.7)
# ---------------------------------------------------------------------------
y_proba = best_clf.predict_proba(X_test)[:, 1]
print("\n--- THRESHOLD TUNING (0.5 to 0.7) ---")
best_f1 = 0.0
best_thresh = 0.5
for t in np.arange(0.5, 0.75, 0.05):
    y_pred_thresh = (y_proba > t).astype(int)
    rec_t = recall_score(y_test, y_pred_thresh)
    prec_t = precision_score(y_test, y_pred_thresh)
    f1_t = f1_score(y_test, y_pred_thresh)
    print(f"Threshold: {t:.2f}, Recall: {rec_t:.3f}, Precision: {prec_t:.3f}, F1: {f1_t:.3f}")
    if f1_t > best_f1:
        best_f1 = f1_t
        best_thresh = t

print(f"\nBest threshold by F1: {best_thresh} (F1={best_f1:.3f})")
y_pred_best = (y_proba > best_thresh).astype(int)
acc_best = accuracy_score(y_test, y_pred_best)
cm_best = confusion_matrix(y_test, y_pred_best)
report_best = classification_report(y_test, y_pred_best)
print(f"\n--- RESULTS AT BEST F1 THRESHOLD = {best_thresh} ---")
print(f"Accuracy: {acc_best:.4f}")
print("Confusion Matrix:")
print(cm_best)
print("\nClassification Report:")
print(report_best)

# ---------------------------------------------------------------------------
# 8. FEATURE IMPORTANCES
# ---------------------------------------------------------------------------
print("\nFeature Importances (XGBoost):")
importances = best_clf.feature_importances_
for feat, imp in zip(features, importances):
    print(f"{feat}: {imp:.3f}")


C:\Users\student\AppData\Local\Temp\ipykernel_13016\1122259737.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("merged_with_wind_and_direction.csv")  # Adjust path if needed


Updated Relative Humidity stats:
count    1.190114e+06
mean     5.465224e+01
std      1.944202e+01
min      5.821928e+00
25%      4.100066e+01
50%      5.368316e+01
75%      6.799662e+01
max      1.000161e+02
Name: RH, dtype: float64

Class distribution (threshold=30.0):
fire_class
0    991642
1    198472
Name: count, dtype: int64

Computed scale_pos_weight = 4.99 (neg:pos)

Starting XGBoost grid search (optimizing F1, with scale_pos_weight)...
Fitting 3 folds for each of 96 candidates, totalling 288 fits
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'scale_pos_weight': 2.4964279195070214, 'subsample': 0.8}

--- RESULTS AT DEFAULT THRESHOLD (0.5) ---
Accuracy: 0.8305
Confusion Matrix:
[[176945  21477]
 [ 18863  20738]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90    198422
           1       0.49      0.52      0.51     39601

    accuracy                  